# CS50 Lecture 4 Key Notes

Week 4 removes the proverbial training wheels — chief among them the CS50 library — and reveals what has been happening underneath the hood of the computer's memory all along. The headline topics: hexadecimal notation, memory addresses, pointers, what a `string` really is, `malloc` and `free`, valgrind, swapping via pass-by-reference, `scanf` and file I/O.

The lecture's own warning applies: pointers are one of the more complicated concepts in C, and understanding tends to arrive with practice rather than on first contact. Nothing here needs to land in one pass.

## 4.1 Images as Bits (Preview)

Any image on a screen is a finite grid of dots called **pixels**, each with a colour.

**Resolution** is simply how many dots run horizontally and how many run vertically. Multiplying the two gives some number of bytes — kilobytes, megabytes or bigger for a massive image.

Zooming far enough into any photograph (the lecture's example: a bowl of stress balls) eventually exposes the individual pixels. The detail is finite; past a point there is nothing more to see.

In the simplest scheme, one bit describes one pixel. A `0` is rendered as black and a `1` as white. An 8×8 grid of such bits is enough to hide a smiley face — the zeros, read as black dots, trace the face:

```text
1 1 0 0 0 0 1 1
1 0 1 1 1 1 0 1
0 1 0 1 1 0 1 0
0 1 1 1 1 1 1 0
0 1 0 1 1 0 1 0
0 1 1 0 0 1 1 0
1 0 1 1 1 1 0 1
1 1 0 0 0 0 1 1
```

A file storing that pattern of zeros and ones, opened in a photos app, is depicted as exactly that grid: an x-and-y arrangement of white and black dots. That is a 1-bit image.

Modern images spend 16 bits, 24 bits or more per colour rather than 1, which is how every colour of the rainbow becomes representable instead of only black and white.

This is a preview: the week's problem set involves writing code that manipulates real image files directly, and §4.16 returns to bitmaps once file I/O is available.

### 4.1.1 Questions

1. A photo is 800 pixels wide and 600 pixels tall, with 24 bits of colour per pixel. How many bytes does the raw image data occupy, and which two facts from this section feed the calculation?
2. In a 1-bit image only two colours are possible. Why, and what change makes millions representable?

## 4.2 Hexadecimal

### 4.2.1 RGB Colour Codes

Recall RGB from Week 0: a colour is described by an amount of red, an amount of green and an amount of blue.

Photoshop's colour picker exposes both notations at once. Black is typed as `000000`, which also reads as 0 red, 0 green, 0 blue. White is `FFFFFF`, equivalently 255 red, 255 green, 255 blue.

With 8 bits per channel the count runs from 0 up to 255 — 255 being the biggest number 8 bits can represent, exactly as in Week 0. Somehow `FF` and 255 are the same number; the next subsection shows why.

| Colour | Hex code | R, G, B     |
|--------|----------|-------------|
| Black  | `000000` | 0, 0, 0     |
| White  | `FFFFFF` | 255, 255, 255 |
| Red    | `FF0000` | 255, 0, 0   |
| Green  | `00FF00` | 0, 255, 0   |
| Blue   | `0000FF` | 0, 0, 255   |

This notation is already familiar from R: a colour like `"#FF0000"` passed to a `ggplot2` scale or `col =` argument is precisely this hexadecimal RGB convention, with `#` as the prefix instead of the `0x` used in C.

### 4.2.2 Counting in Base 16

Binary offers 2 digits, decimal 10. **Hexadecimal** (aka **base-16**) offers 16: the Arabic numerals run out at 9, so the letters A through F are drafted in, upper or lower case.

| Hex digit | Decimal value |
|-----------|---------------|
| `0`–`9`   | 0–9           |
| `A`       | 10            |
| `B`       | 11            |
| `C`       | 12            |
| `D`       | 13            |
| `E`       | 14            |
| `F`       | 15            |

An infinite number of base systems exists (base 3, base 15, base 17…); hexadecimal is merely one of the few conventions popular in computing.

The positional system itself is unchanged — same columns, same placeholders — only the column weights differ. In a two-digit hexadecimal number the right column is the 1s place (16⁰ = 1) and the left column the 16s place (16¹ = 16).

Counting therefore proceeds: `00`, `01`, `02` … `09`, then — where decimal would carry the 1 — hexadecimal keeps going: `0A`, `0B`, `0C`, `0D`, `0E`, `0F`. Only then does the carry happen, producing `10`.

That `10` is *not* ten. It is 16 × 1 + 1 × 0 = 16. From `00` up through `0F` is a total of 16 combinations, which is exactly why the carry lands there.

The highest two-digit value is `FF`: 16 × 15 + 1 × 15 = 240 + 15 = **255**. Hence the pairs of Fs in the colour picker — `FF` is how 255 is written in hexadecimal.

### 4.2.3 One Hex Digit Is Four Bits

The real reason hexadecimal is convenient: a single hex digit represents exactly 4 bits.

`F` is 15, and 15 in binary is `1111` — one in the 8s place, plus one in the 4s, plus one in the 2s, plus one in the 1s.

A byte is 8 bits, so two hex digits describe one byte perfectly. `FF` is `1111 1111`, i.e. 128 + 64 + 32 + 16 + 8 + 4 + 2 + 1 = 255. Grouping bits into clusters of 4, each cluster covers all possibilities from 0 through 15 using just `0` through `F`.

That clean one-digit-per-four-bits mapping is the entire justification: it is why the computing world reaches for hex when talking about colours — and, as this lecture shows, memory.

### 4.2.4 The 0x Prefix

A bare `10` on a whiteboard is ambiguous: byte ten, or byte sixteen? Without knowing the base system in play, the numeral alone cannot say.

The convention is to prefix every hexadecimal number with `0x`. The characters `0` and `x` mean nothing by themselves; they simply announce that what follows is base-16. `0x10` is unambiguously the number 16.

> The arithmetic rarely matters in practice. What matters is *recognising* the notation: this week is full of `0x` followed by two, four or eight hex digits, and generally nobody bothers translating them to decimal.

### 4.2.5 Cheat Sheet — Number Systems

| System      | Base | Digits              | Example  | Decimal value        |
|-------------|------|---------------------|----------|----------------------|
| Binary      | 2    | `0 1`               | `1111`   | 15                   |
| Decimal     | 10   | `0`–`9`             | `255`    | 255                  |
| Hexadecimal | 16   | `0`–`9`, `A`–`F`    | `0xFF`   | 255                  |

- 1 hex digit ↔ 4 bits; 2 hex digits ↔ 1 byte.
- `0x` prefix = "this number is hexadecimal".
- `FF` = 16 × 15 + 15 = 255 — the per-channel maximum in RGB.

### 4.2.6 Questions

1. Converting `0xA5` to decimal, with the place values written out — what number results?
2. Why does one hexadecimal digit correspond to exactly 4 bits?
3. The colour channel value `0x80` is what in decimal, and roughly what fraction of full intensity?
4. Why is hexadecimal `10` sixteen rather than ten, and what convention removes the ambiguity in writing?
5. "Full red, half green, no blue" — what 6-digit hex code expresses it?

## 4.3 Memory Addresses

The canvas of memory drawn in previous weeks — a grid of bytes — was numbered 0, 1, 2, 3 … in decimal. Nothing wrong with that, but any real programmer numbers those locations in hexadecimal instead, purely because of the 4-bits-per-digit convenience above.

The bytes therefore run `0` through `9`, then `A` through `F`, then `10`, `11` … `19`, `1A`, `1B` … `1F` and so on — or, unambiguously, `0x0` through `0x1F`.

A concrete starting point, `addresses0.c` — nothing new yet, literally just declaring a variable and printing its value:

```c
// Prints an integer

#include <stdio.h>

int main(void)
{
    int n = 50;
    printf("%i\n", n);
}
```

```text
$ make addresses
$ ./addresses
50
```

What matters is where that `n` ends up. Somewhere in the grid of memory, 4 bytes are set aside — an `int` is 4 bytes, aka 32 bits — and the 50 is stored there. Other parts of memory may already be in use by the program, so the spot is essentially arbitrary.

For the sake of every diagram that follows, that address is pretended to be `0x123` — easily pronounceable, though a real address is a much larger number. When `printf` is handed `n`, the computer goes to that location and prints what it finds. Programs have been doing exactly this for weeks; it was simply invisible.

### 4.3.1 Questions

1. `int n = 50;` — how many bytes are set aside, and by what convention?
2. Why do programmers number memory locations in hexadecimal rather than decimal?
3. What does the computer actually do when `printf("%i\n", n)` runs?

## 4.4 Pointers

### 4.4.1 Two New Operators: & and *

Two new pieces of syntax unlock everything this week:

| Operator | Name | Meaning |
|----------|------|---------|
| `&`  | address-of operator | yields the address at which a variable is stored |
| `*` (after a type, in a declaration) | pointer declaration | the variable holds the *address of* a value of that type |
| `*` (before a pointer, in an expression) | dereference operator | go *to* the address stored in the pointer |

The ampersand has one straightforward job: prefixing a variable with `&` asks the computer at what address that variable lives. `&n` reads as "the address of `n`". It works for any data type, not just `int` — strings included, as §4.5 shows.

The asterisk is the overloaded one — the same symbol already means multiplication, and now it means two further things depending on context. The developers of C decades ago chose one symbol for several ideas; the ambiguity is acknowledged even by seasoned programmers as cognitively confusing, and it simply has to be lived with.

### 4.4.2 Printing an Address: %p

`addresses1.c` swaps the value for the address:

```c
// Prints an integer's address

#include <stdio.h>

int main(void)
{
    int n = 50;
    printf("%p\n", &n);
}
```

An address is technically just a number, but the conventional format code for printing one is `%p`, not `%i`. The output is something like:

```text
0x7ffd3c34ecc
```

All hexadecimal digits — a real address, far bigger than `0x123`, because the machine has far more memory than that toy address suggests. Doing the mental math to decimal would be painful and is never necessary; the `%p` view is demonstrative rather than routinely useful.

### 4.4.3 Declaring a Pointer

A **pointer** is a variable that stores an address.

An address is technically just a number, but a deliberate distinction is drawn between integers that are *data* (like 50, things worth doing math on) and pointers, whose value is the address of some other value in memory.

`addresses2.c` stores the address before printing it:

```c
// Stores and prints an integer's address

#include <stdio.h>

int main(void)
{
    int n = 50;
    int *p = &n;
    printf("%p\n", p);
}
```

The declaration `int *p` reads: `p` is a variable that stores *the address of an int* — not an `int` itself. The `&n` on the right supplies that address. Printing `p` shows the same big hexadecimal number as before.

The asterisk placement is pure convention. All three of these compile identically:

```c
int *p = &n;   // canonical CS50 style: space, then star attached to the name
int* p = &n;   // star attached to the type
int * p = &n;  // star floating in the middle
```

The first form is the recommended style, even though attaching the star to `int` would arguably better express that the star modifies the type rather than the name.

> Omitting the asterisk is an error, not a style choice. `int p = &n;` fails to compile with **"incompatible pointer to integer conversion"** — a pointer on the right cannot be stored in a plain `int` on the left, even though both are numbers at the end of the day. The fix is simply to restore the `*`.

### 4.4.4 Why a Pointer Is 8 Bytes

On most modern systems a pointer occupies **8 bytes** (64 bits) — twice the size of the `int` it might point to.

The reason is counting range. A 32-bit pointer can only count to about 4bn, and 4bn addresses is 4 gigabytes of addressable memory. Computers routinely carry 8, 16 or tens of gigabytes, so 32 bits could not address it all; 64-bit pointers can.

### 4.4.5 Mailboxes, Arrows and the Foam Finger

A physical mental model: the computer's memory is hundreds or thousands of little mailboxes, apartment-style, in rows and columns.

One mailbox (labelled `n`, at address `0x123`) contains a useful value, 50. Another mailbox (labelled `p`) contains not a value but *the address* `0x123` — the house number of the first mailbox. Following what `p` says — like a giant foam finger pointing across the room — leads to the mailbox where the 50 actually lives.

In practice nobody cares what the specific addresses are. On a whiteboard, in section, in office hours, the address is abstracted away entirely and the pointer is drawn as an **arrow**: `p` simply points at the 50. That arrow picture is the working mental model from here on.

> An aside for R users: R has no pointer syntax at all. Every name in R behaves like a value, and modifying one object does not silently modify another, thanks to copy-on-modify semantics. C exposes the address layer that R deliberately hides — which buys control and speed, at the price of every bug in the rest of this lecture.

### 4.4.6 Dereferencing: Going to the Address

The second use of `*`: prefixing a *pointer* with a star means "go to the address stored in here". That is the **dereference operator**.

`addresses3.c` prints the 50 *via* the pointer:

```c
// Stores and prints an integer via its address

#include <stdio.h>

int main(void)
{
    int n = 50;
    int *p = &n;
    printf("%i\n", *p);
}
```

```text
$ make addresses
$ ./addresses
50
```

Line by line, what the CPU does:

| Line | Code | Effect |
|------|------|--------|
| 1 | `int n = 50;` | 4 bytes are set aside (say at `0x123`) and 50 is stored there |
| 2 | `int *p = &n;` | 8 bytes are set aside for `p`, and the address `0x123` is stored there |
| 3 | `printf("%i\n", *p);` | `*p` follows the arrow: go to `0x123`, fetch what lives there (50), print it |

Printing `n` directly would of course show the same thing; the point is that the value is now reachable through its address alone.

### 4.4.7 Cheat Sheet — Pointer Syntax

| Syntax | Context | Reads as |
|--------|---------|----------|
| `&x`     | expression | "the address of `x`" |
| `int *p` | declaration | "`p` stores the address of an `int`" |
| `*p`     | expression | "go to the address stored in `p`" |
| `%p`     | format code | print an address |

- A pointer is a variable that stores an address; on modern systems it is 8 bytes.
- Same syntax for every type: `char *`, `float *` etc. all declare pointers the same way.
- Whiteboard convention: forget the hex, draw an arrow.

### 4.4.8 Questions

1. In one sentence: what is a pointer?
2. In `int *p = &n;`, what do `int *`, `p` and `&n` each contribute?
3. Why are pointers 8 bytes on modern systems, when 4 bytes once sufficed?
4. Given `int n = 50; int *p = &n;`, what does `printf("%i\n", *p);` print, and by what steps?
5. What happens at compile time with `int p = &n;`, and why is it rejected even though an address is "just a number"?
6. In the mailbox picture: what plays the role of the mailbox label, the slip of paper inside pointer-mailbox `p`, and the foam finger?

## 4.5 Strings Unmasked: char *

### 4.5.1 The White Lie

Since Week 1, `string` has been used as if it were a real C type. It is not. C has no data type spelled `s`-`t`-`r`-`i`-`n`-`g`; the concept exists, the keyword does not.

The familiar starting point, `addresses4.c` (training wheels still on):

```c
// Prints a string

#include <cs50.h>
#include <stdio.h>

int main(void)
{
    string s = "HI!";
    printf("%s\n", s);
}
```

In memory, the string is exactly what Week 2 said: an array of characters, back to back, ending in the null character:

```text
+-------+-------+-------+-------+
|   H   |   I   |   !   |  \0   |
+-------+-------+-------+-------+
 0x123   0x124   0x125   0x126
```

Three visible characters, **4 bytes** — always one extra for the terminator. Because a string is a sequence of characters back to back to back, the addresses are necessarily **contiguous**: no gaps.

### 4.5.2 What s Actually Stores

The question the course dodged in Week 1: `s` itself must live somewhere in memory too — so what value is *in* it?

Not the characters. `s` stores **the address of the first character** — here, `0x123`. That is the whole trick:

- the **pointer** tells the computer where the string *begins*;
- the **null terminator** (`\0`) tells it where the string *ends*;
- everything in between is findable with a loop.

That is how something as interesting as a string is built out of nothing but numbers. The address of the first character is sufficient precisely because of the `\0` convention agreed decades ago — no need to store the address of every character.

`addresses5.c` makes it visible:

```c
// Prints a string's address as well the addresses of its chars

#include <cs50.h>
#include <stdio.h>

int main(void)
{
    string s = "HI!";
    printf("%p\n", s);
    printf("%p\n", &s[0]);
    printf("%p\n", &s[1]);
    printf("%p\n", &s[2]);
    printf("%p\n", &s[3]);
}
```

Sample output (real run):

```text
0x561999bd0004   ← s itself
0x561999bd0004   ← &s[0] — identical
0x561999bd0005   ← &s[1] — one byte along
0x561999bd0006   ← &s[2] — one more
0x561999bd0007   ← &s[3] — the null terminator's address
```

`s` and `&s[0]` print the same address, and each successive character sits exactly one byte later.

> Why does `s[0]` need an ampersand when `s` does not? Because `s` *already is* an address — the address of the whole string, which by design equals the address of its first character. `s[0]`, by contrast, is a `char` (a value), so getting its location requires the address-of operator.

### 4.5.3 typedef and the string Synonym

The reveal: in the CS50 library, `string` has been, since Week 1, a synonym created with `typedef`:

```c
typedef char *string;
```

`typedef` defines a new type name. Week 3's phone book already used it, bundled with a `struct`:

```c
typedef struct
{
    string name;
    string number;
} person;
```

The `struct` part was specific to the phone book; `typedef` alone is more generally useful. A pointless-but-legal example — a synonym for `int`:

```c
typedef int integer;
```

After that line, `integer` is literally equivalent to `int`. The CS50 library does the same thing, telling the compiler that `char *` may be spelled `string` — purely so that Weeks 1–3 never had to mention hexadecimal, addresses, pointers or dereferencing.

So `string s` has really meant `char *s` all along: *`s` is the address of a `char`* — the first `char` of the string.

Removing the training wheel is mechanical. With `#include <cs50.h>` deleted, compiling `string s = "HI!";` fails with **"use of undeclared identifier 'string'"**. Renaming the type fixes it, as in `addresses7.c`:

```c
// Declares a string without CS50 Library

#include <stdio.h>

int main(void)
{
    char *s = "HI!";
    printf("%s\n", s);
}
```

Raw C, no CS50 scaffolding, same output.

### 4.5.4 How printf Handles %s

Handing `printf` the address `s` works because of the format code. Given `%s`, `printf` goes to that address and prints character after character — a loop inside its implementation — until it hits the null terminator, then stops.

Two consequences:

- No `*` is needed when printing a string. `printf` does the dereferencing internally; and conceptually the *address* is exactly what `%s` wants, since a string is identified by where it begins.
- `%c` is the opposite request: print a single character, no loop, stop after one.

### 4.5.5 Questions

1. `char *s = "HI!";` — what single value is stored in `s`, and where do the characters live?
2. Why is knowing only the first character's address enough to recover an entire string?
3. What does `typedef char *string;` do, and why did the CS50 library include it?
4. Why does `&s[0]` need the ampersand when `s` alone does not?
5. How does `printf`'s `%s` know when to stop printing, and what does `%c` do differently?

## 4.6 Pointer Arithmetic

### 4.6.1 Characters via Offsets

Addresses are numbers, so math on them is possible: given an address, adding 1 yields the next address. That is **pointer arithmetic**. (Multiplying or dividing addresses would be weird; adding and subtracting is the useful part.)

Week 2's array notation, `addresses8.c`:

```c
// Prints a string's chars

#include <stdio.h>

int main(void)
{
    char *s = "HI!";
    printf("%c\n", s[0]);
    printf("%c\n", s[1]);
    printf("%c\n", s[2]);
}
```

The same program with the new vocabulary, `addresses9.c`:

```c
// Prints a string's chars via pointer arithmetic

#include <stdio.h>

int main(void)
{
    char *s = "HI!";
    printf("%c\n", *s);
    printf("%c\n", *(s + 1));
    printf("%c\n", *(s + 2));
}
```

`*s` reads: `s` is the address of the first character, so going there lands on `H` by definition. `*(s + 1)` moves one character along, then goes there — the parentheses keep the order of operations honest, as in math class. Output in both versions: `H`, `I`, `!`.

### 4.6.2 Substrings for Free

Handing `%s` an address partway into a string prints from there to the terminator, `addresses10.c`:

```c
// Prints substrings via pointer arithmetic

#include <stdio.h>

int main(void)
{
    char *s = "HI!";
    printf("%s\n", s);
    printf("%s\n", s + 1);
    printf("%s\n", s + 2);
}
```

```text
HI!
I!
!
```

All three calls work because all three "strings" happen to be terminated by the same null character. A string is, literally, whatever sequence of characters starts at the given address and runs to the next `\0`.

### 4.6.3 Array Notation Is Syntactic Sugar

`s[i]` and `*(s + i)` are identical; the bracket form is **syntactic sugar** — nicer syntax for the same operation. The computer converts the bracket notation into the pointer-arithmetic version. Nobody wants to read `*(s + 2)` when `s[2]` is available, but the brackets were the abstraction all along, not the reality. (The sugar runs in both directions — §4.14.3 uses an array where a pointer is expected.)

### 4.6.4 Questions

1. `s[2]`, rewritten without brackets?
2. For `char *s = "HI!";`, the output of `printf("%s\n", s + 1);`?
3. Why are the parentheses in `*(s + 1)` not optional?
4. What does "syntactic sugar" mean, with the bracket notation as the example?

## 4.7 Comparing Strings

### 4.7.1 Integers Compare Cleanly

Comparing two `int`s with `==` has worked since Week 1, `compare0.c`:

```c
// Compares two integers

#include <cs50.h>
#include <stdio.h>

int main(void)
{
    // Get two integers
    int i = get_int("i: ");
    int j = get_int("j: ");

    // Compare integers
    if (i == j)
    {
        printf("Same\n");
    }
    else
    {
        printf("Different\n");
    }
}
```

Typing 50 and 50 prints `Same`. Two `int` variables store their values directly, so `==` compares exactly what it appears to compare.

### 4.7.2 == on Strings Compares Addresses

The same pattern with strings, `compare1.c`:

```c
// Compares two strings' addresses

#include <cs50.h>
#include <stdio.h>

int main(void)
{
    // Get two strings
    char *s = get_string("s: ");
    char *t = get_string("t: ");

    // Compare strings' addresses
    if (s == t)
    {
        printf("Same\n");
    }
    else
    {
        printf("Different\n");
    }
}
```

Typing `HI!` for both prompts still prints `Different` — the Week 2 mystery, now explicable.

Each call to `get_string` allocates its **own** chunk of memory for whatever the human typed, even when the text is identical; no cleverness reuses the first copy. So `HI!\0` lands at, say, `0x123`–`0x126` for `s`, and a second `HI!\0` lands at `0x456`–`0x459` for `t`:

```text
 s = 0x123                     t = 0x456
   |                             |
   v                             v
+---+---+---+----+           +---+---+---+----+
| H | I | ! | \0 |           | H | I | ! | \0 |
+---+---+---+----+           +---+---+---+----+
0x123 ...    0x126           0x456 ...    0x459
```

`s == t` therefore asks "does `0x123` equal `0x456`?" — and the answer, correctly, is no. The comparison was never wrong; it was answering a different question than intended. The intent was to compare the *characters*, not the addresses thereof.

### 4.7.3 strcmp Compares Characters

`strcmp` (from `string.h`) exists precisely for this. Written decades ago, it contains its own loop that starts at the beginning of each string and compares character by character by character. `compare2.c`:

```c
// Compares two strings using strcmp

#include <cs50.h>
#include <stdio.h>
#include <string.h>

int main(void)
{
    // Get two strings
    char *s = get_string("s: ");
    char *t = get_string("t: ");

    // Compare strings
    if (strcmp(s, t) == 0)
    {
        printf("Same\n");
    }
    else
    {
        printf("Different\n");
    }
}
```

Now `hi` vs `hi` prints `Same`, and `hi` vs `by` prints `Different`.

> Per the documentation, `strcmp` returns **0** when the strings are equal — not 1, not `true`. (Its other return values signal which string sorts before or after the other.) Testing `strcmp(s, t) == 0` is the idiom; the Week 2 problem sets already relied on it.

R hides this entire problem: `"hi" == "hi"` in R compares the contents, because R strings are values, not addresses. C's `==` is the lower-level operation that R's `==` is built on top of.

### 4.7.4 Seeing the Two Addresses

Printing the pointers themselves (`compare3.c` prints the strings; `compare4.c` prints their addresses) makes the two allocations visible:

```c
// Prints two strings' addresses

#include <cs50.h>
#include <stdio.h>

int main(void)
{
    // Get two strings
    char *s = get_string("s: ");
    char *t = get_string("t: ");

    // Print strings' addresses
    printf("%p\n", s);
    printf("%p\n", t);
}
```

Typing the exact same `hi` twice yields two different addresses — in the lecture's run, subtly different: one ending in `b0`, the other in `f0`. Two chunks of memory, two pointers, same text.

(Where the computer *chooses* to put each allocation is deliberate and often auto-incremented — back to back when possible, messier over time as larger programs add and remove values. §4.12.3 lays out the map.)

### 4.7.5 Questions

1. Two identical strings are typed, yet `if (s == t)` prints `Different` — what is actually being compared?
2. What does `strcmp` return for equal strings, and what do its other return values convey?
3. Why does `get_string` produce two separate chunks of memory for the same text typed twice?
4. Under what circumstance is `s == t` *true* for two `char *` variables?

## 4.8 Copying Strings, malloc and free

### 4.8.1 The Accidental Alias

Copying an `int` is one assignment. Copying a string the same way goes quietly wrong, `copy0.c`:

```c
// Capitalizes a string

#include <cs50.h>
#include <ctype.h>
#include <stdio.h>
#include <string.h>

int main(void)
{
    // Get a string
    string s = get_string("s: ");

    // Copy string's address
    string t = s;

    // Capitalize first letter in string
    t[0] = toupper(t[0]);

    // Print string twice
    printf("s: %s\n", s);
    printf("t: %s\n", t);
}
```

Typing `hi!` in lowercase prints:

```text
s: Hi!
t: Hi!
```

**Both** got capitalised, although only `t[0]` was touched. The reason is now mechanical: `s` is an address, so `t = s` copies *the address*, not the characters. If `s` holds `0x123`, then `t` now also holds `0x123` — two arrows to the same chunk of memory:

```text
   s ────────┐
             v
        +---+---+---+----+
        | h | i | ! | \0 |
        +---+---+---+----+
             ^
   t ────────┘
```

`t[0]` — equivalently `*t`, without the sugar — is therefore the exact same byte as `s[0]`. Capitalising one capitalises "both", because there is only one.

> The R contrast is the sharpest bridge here. In R, `t <- s` behaves like a copy: modifying `t` never changes `s`, because R copies on modification. The C line that *looks* identical produces an alias instead. What R does automatically must be done by hand in C — which is what the rest of this section is for.

Two library functions from `stdlib.h` make a real copy possible:

- **`malloc(n)`** — *memory allocation*: asks the operating system for `n` bytes of memory and returns the **address of the first byte** of that chunk. Where the chunk lives is not the caller's concern.
- **`free(p)`** — the opposite: hands a previously `malloc`'d chunk back to the computer for reuse.

### 4.8.2 A Manual Copy

`copy2.c` allocates a second chunk and copies character by character:

```c
// Capitalizes a copy of a string

#include <cs50.h>
#include <ctype.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int main(void)
{
    // Get a string
    char *s = get_string("s: ");

    // Allocate memory for another string
    char *t = malloc(strlen(s) + 1);

    // Copy string into memory, including '\0'
    for (int i = 0; i <= strlen(s); i++)
    {
        t[i] = s[i];
    }

    // Capitalize copy
    t[0] = toupper(t[0]);

    // Print strings
    printf("s: %s\n", s);
    printf("t: %s\n", t);
}
```

Two details carry all the weight:

- **`malloc(strlen(s) + 1)`** — `strlen` reports the human-visible length (3 for `hi!`), but the copy needs one extra byte for the `\0`. Hard-coding `malloc(4)` would work for `hi!` and break for everything else.
- **`i <= strlen(s)`** — the `<=` (rather than the usual `<`) makes the loop run one extra iteration, copying the null terminator itself. With `<`, the copy would have no terminator; manually appending `t[3] = '\0'` (or worse, hard-coding a 4) would be sloppier fixes for the same omission.

Now `hi!` prints as:

```text
s: hi!
t: Hi!
```

The original survives; only the copy is capitalised. `t` points at its own 4 bytes — freshly allocated, initially garbage, then overwritten `h`, `i`, `!`, `\0` one iteration at a time.

### 4.8.3 The Design Flaw: strlen in the Condition

`copy2.c` still has a design problem seen in Week 2: the loop condition calls `strlen(s)` on *every* iteration, asking the same question again and again wastefully.

The fix, `copy3.c`, computes the length once in the initialisation:

```c
    // Copy string into memory, including '\0'
    for (int i = 0, n = strlen(s); i <= n; i++)
    {
        t[i] = s[i];
    }
```

`n` is set a single time; each iteration then compares `i` against a stored number instead of re-walking the string.

### 4.8.4 strcpy

The manual loop is such a common need that C ships a function for it: `strcpy` (from `string.h`). `copy4.c`:

```c
    // Copy string into memory
    strcpy(t, s);
```

> Argument order: **destination first, then source** — it can feel backwards, and mixing them up is a classic mistake. Per the documentation: `strcpy(dst, src)`.

`strcpy` does the same work the `for` loop did, terminator included, provided enough memory was allocated at the destination beforehand.

### 4.8.5 NUL versus NULL

Two similarly-named things, deliberately distinguished:

| Name | What it is | Value |
|------|------------|-------|
| **NUL**  | the null *character*, `\0` — the string terminator | one byte, all 8 bits zero |
| **NULL** | a special *address* at which nothing is ever stored | address `0x0` |

Humans decades ago decided to waste byte location 0 (and a few bytes after it) on purpose: never put anything there, so that the address `0x0` can serve as a sentinel meaning *"something went wrong"*.

That sentinel matters because functions that hand back memory can fail:

- **`get_string` can return `NULL`** — e.g. if the human types in such a large paragraph that no room remains in memory. Rather than silently returning half the text, it returns `NULL`: "cannot oblige".
- **`malloc` can return `NULL`** — no memory available.

Proceeding to *use* a `NULL` as if it were a real address means touching `0x0`, which is exactly the kind of invalid access §4.11 dramatises.

### 4.8.6 The Fully Defensive Version

`copy5.c` adds every check that has technically been owed since Week 1 (consciously skipped back then as too much overhead):

```c
// Capitalizes a copy of a string without memory errors

#include <cs50.h>
#include <ctype.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int main(void)
{
    // Get a string
    char *s = get_string("s: ");
    if (s == NULL)
    {
        return 1;
    }

    // Allocate memory for another string
    char *t = malloc(strlen(s) + 1);
    if (t == NULL)
    {
        return 1;
    }

    // Copy string into memory
    strcpy(t, s);

    // Capitalize copy
    if (strlen(t) > 0)
    {
        t[0] = toupper(t[0]);
    }

    // Print strings
    printf("s: %s\n", s);
    printf("t: %s\n", t);

    // Free memory
    free(t);
    return 0;
}
```

Each guard, in order:

| Check | Failure it prevents |
|-------|---------------------|
| `s == NULL` | `get_string` failed; using `s` would dereference `0x0` |
| `t == NULL` | `malloc` failed; same hazard |
| `strlen(t) > 0` | the human typed only Enter — a valid *empty* string; `t[0]` would then be the `\0` itself, which makes no sense to capitalise and at worst touches memory it should not |
| `free(t)` at the end | the memory leak described next |
| `return 1` / `return 0` | explicit error/success signalling, as in earlier weeks |

### 4.8.7 Memory Leaks

`malloc`'d memory is never given back automatically while the program runs. Failing to `free` it is a **memory leak**.

In a program this short it hardly matters — the operating system reclaims everything at exit. In long-running programs (servers, apps left open for days) leaks accumulate: the computer believes ever more of its memory is in use, and the program — a Mac app, a Windows app, an iPhone or Android app that somehow gets slower and slower and slower — is often exhibiting exactly this symptom of a human having messed up.

The rule of thumb:

> **Whatever was malloc'd must be freed — by whoever malloc'd it.** Memory obtained from `get_string` is the exception: the CS50 library frees its own allocations automatically when no longer needed (a deliberate bit of magic that disappears along with the library). Hence: no `free(s)`, but definitely `free(t)`.

In R terms: `free` is what the garbage collector does invisibly after `gc()` or whenever needed. C has no collector; the bookkeeping is manual.

### 4.8.8 Questions

1. After `char *t = s;` followed by `t[0] = toupper(t[0]);`, both `s` and `t` print capitalised — by what mechanism?
2. Why `malloc(strlen(s) + 1)` rather than `malloc(strlen(s))`?
3. In the copying loop, why `i <= strlen(s)` instead of the usual `<`?
4. That same loop condition hides a second, non-fatal flaw — what is it, and what is the fix?
5. The argument order of `strcpy`, and the consequence of reversing it?
6. NUL versus NULL — what is each?
7. What is a memory leak, what everyday symptom does it produce, and why is `free(s)` wrong for a `get_string` string while `free(t)` is required after `malloc`?

## 4.9 Valgrind

**valgrind** is a memory-error checker — a complement to `printf`, debug50 and the duck, specialised for chasing memory-related mistakes. It is a standard programmers' tool, not a CS50 one.

The deliberately buggy specimen, `memory.c`:

```c
// Demonstrates memory errors via valgrind

#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int *x = malloc(3 * sizeof(int));
    x[1] = 72;
    x[2] = 73;
    x[3] = 33;
}
```

One new operator first: **`sizeof`** asks how big a data type is on this specific system. `sizeof(int)` is usually 4; `sizeof(char)` is always 1. Writing `malloc(3 * sizeof(int))` instead of a hard-coded `malloc(12)` keeps the code portable to systems where an `int` is not 4 bytes.

The line `int *x = malloc(...)` allocates 12 bytes and stores the address of the first byte in `x`. Bracket notation then works on that chunk — the syntactic sugar again. Equivalently, with pointer arithmetic:

```c
*x = 72;
*(x + 1) = 73;
*(x + 2) = 33;
```

Identical behaviour; most people use the brackets because they are cleaner to read and write.

Three bugs hide in `memory.c`:

1. **Indexing is off.** Space for 3 `int`s means valid indices 0, 1 and 2 — not 1, 2 and 3. `x[3]` touches memory beyond the allocation.
2. **No `free(x)`.** A memory leak.
3. **No `NULL` check** on `malloc`'s return (not flagged by valgrind here because the allocation happens to succeed, but owed on principle per §4.8.5).

Compiling and running normally shows *nothing wrong* — no compiler error, no crash. The bug is latent: the program is simply not getting unlucky enough to crash. That is exactly the class of bug valgrind exists for:

```text
$ valgrind ./memory
```

The output is atrocious at a glance — verbose, a process number repeated on every line — but two lines are juicy once the eye is trained:

| valgrind says | Translation |
|---------------|-------------|
| `Invalid write of size 4` … line 11 | somewhere, 4 bytes (an `int`!) are being *written* to memory that was never allocated — line 11 is `x[3] = 33;` |
| `definitely lost: 12 bytes in 1 blocks` … allocated at line 8 | 12 bytes = 3 × 4 = the whole `malloc` from line 8 was never freed |

("Write" means changing a value; "read" means accessing one — the messages distinguish the two.) The numbers are breadcrumbs: knowing an `int` is 4 bytes turns "size 4" and "12 bytes" into pointers straight at the guilty lines.

The fixed version — indices `0`/`1`/`2`, plus `free(x)` at the end, plus the `NULL` check with `return 1` — re-run under valgrind reports:

```text
All heap blocks were freed -- no leaks are possible
ERROR SUMMARY: 0 errors from 0 contexts
```

"Heap" gets defined in §4.12.3. valgrind remains among the most arcane tools in the course, but the habit is simple: look for a file name and line number in the noise, and follow it.

### 4.9.1 Questions

1. Why is `malloc(3 * sizeof(int))` preferred over `malloc(12)`?
2. In `memory.c`, what does valgrind's "Invalid write of size 4" at line 11 point to, and why size 4?
3. What does "definitely lost: 12 bytes in 1 blocks" mean, and where does the 12 come from?
4. `./memory` ran without any visible error — why is that not evidence of correctness?

## 4.10 Garbage Values

Memory handed to a program is not necessarily empty. Whatever the previous occupant left behind is still physically there; uninitialised variables therefore contain **garbage values** — remnants of earlier activity, not values the current code put there.

`garbage.c` prints 1024 of them:

```c
#include <stdio.h>
#include <stdlib.h>

int main(void)
{
    int scores[1024];
    for (int i = 0; i < 1024; i++)
    {
        printf("%i\n", scores[i]);
    }
}
```

The array is declared but never filled — no `get_int`, no manual scores. The loop prints whatever happens to occupy those 4096 bytes: some zeros, a 25, a 32000-ish value, negative numbers — noise from previous computation.

> The takeaway is not to *read tea leaves* in garbage; it is that uninitialised memory must not be trusted or touched. Where the garbage comes from becomes clear in §4.12.3: stack memory is constantly reused as functions come and go.

For contrast, R pre-initialises: `numeric(1024)` is guaranteed to be 1024 zeros. C skips that courtesy for speed, so initialisation is the programmer's job.

### 4.10.1 Questions

1. What is a garbage value, and where do the 1024 numbers printed by `garbage.c` actually come from?
2. What does R's `numeric(1024)` guarantee that C's `int scores[1024];` does not?

## 4.11 Binky: Dereferencing an Invalid Pointer

Garbage in an `int` is unfortunate. Garbage in a *pointer* is dangerous, because dereferencing it means jumping to a random address. The lecture's claymation (Nick Parlante's *Pointer Fun with Binky*, Stanford) animates this exact program:

```c
int main(void)
{
    int *x;
    int *y;

    x = malloc(sizeof(int));

    *x = 42;
    *y = 13;   // ← Binky's demise

    y = x;

    *y = 13;   // fine now
}
```

Declaring `int *x; int *y;` without `=` is legal — a variable need not be initialised at declaration — but until assignment each holds a garbage address.

Step by step:

| Step | Code | Effect |
|------|------|--------|
| 1 | `x = malloc(sizeof(int));` | `x` now points at a valid 4-byte chunk (a **pointee**, in the video's vocabulary) |
| 2 | `*x = 42;` | go to that chunk, store 42 — valid |
| 3 | `*y = 13;` | `y` was never pointed at anything; this dereferences a garbage address — **bad things happen** (crash, corruption) |
| 4 | `y = x;`  | pointer assignment: `y` now points at the *same* pointee as `x`; no values copied |
| 5 | `*y = 13;` | valid — and because `x` and `y` share the pointee, the 42 becomes 13 for both |

The claymation's framing is worth keeping: allocating a *pointer* and setting up its *pointee* are **separate steps**, and the second is the one that gets forgotten.

> With great power comes great responsibility. C is so close to the hardware — such fine-grained control over memory — that really fast code is possible, which is why C remains omnipresent decades later. The same control is why so much of today's hacked or crashing software traces back to some human missing a simple memory mistake like this one. More modern languages (Python, arriving in two weeks; Java) build in defences that make such bugs largely impossible — and pay for it with reduced control and, often, speed.

### 4.11.1 Questions

1. `*x = 42;` succeeds while `*y = 13;` is fatal — what is the difference between the two pointers at that moment?
2. Does `y = x;` copy the 42? What does it do?
3. After the fix, storing 13 through `y` changes what is seen through `x` — why?
> It shares the same address

## 4.12 Swapping Two Values

### 4.12.1 The Temporary Variable

The physical setup: two glasses, one holding blue liquid, one holding orange, poured "wrong" — and the challenge of swapping their contents without mixing them.

With only the two glasses the task is impossible: neither liquid has anywhere to go. With a third, *empty* glass it takes three pours:

```text
1. blue   → temp glass      (temp = a)
2. orange → blue's glass    (a = b)
3. temp   → orange's glass  (b = temp)
```

Swapping two values requires a temporary variable. That is the takeaway, and it translates literally into code.

### 4.12.2 A Swap That Fails

`swap0.c` is the faithful translation — and it does not work:

```c
// Fails to swap two integers

#include <stdio.h>

void swap(int a, int b);

int main(void)
{
    int x = 1;
    int y = 2;

    printf("x is %i, y is %i\n", x, y);
    swap(x, y);
    printf("x is %i, y is %i\n", x, y);
}

void swap(int a, int b)
{
    int tmp = a;
    a = b;
    b = tmp;
}
```

```text
x is 1, y is 2
x is 1, y is 2
```

The logic of `swap` is correct — `tmp` is the empty glass — yet `x` and `y` emerge untouched.

The culprit is **scope**, previously described colloquially as "a variable exists only inside the most recent curly braces". `a` and `b` live inside `swap`'s braces; `x` and `y` inside `main`'s. They are different variables.

More precisely: in C, arguments are always **passed by value** — the function receives *copies*. Swapping the copies works perfectly and changes nothing outside.

> R behaves the same way here: a function body that reassigns its arguments (`swap <- function(a, b) { tmp <- a; a <- b; b <- tmp }`) has no effect on the caller's variables either. The difference is what comes next — C offers an escape hatch, and (outside exotic tricks) R does not.

### 4.12.3 The Memory Layout of a Running Program

Why copies land in different places requires the full map of a program's memory. When a program is run — double-clicked on a Mac or PC, tapped on a phone — its compiled zeros and ones are copied from persistent storage into RAM, and the program's memory is organised by convention like this:

```text
+----------------------+
|     machine code     |  ← the program's own compiled 0s and 1s
+----------------------+
|       globals        |  ← variables declared outside main / any function
+----------------------+
|        heap          |  ← malloc allocates from here
|          ↓           |     (grows downward)
|                      |
|          ↑           |
|        stack         |  ← function calls + local variables
+----------------------+     (grows upward)
```

- **Machine code** sits at one edge of the program's memory.
- **Global variables** — defined outside `main`, unused in this course so far — sit just below, accessible everywhere.
- The **heap** is `malloc`'s territory. Successive `malloc` calls hand out chunk after chunk, back to back, working downward. The caller never worries where exactly the memory comes from; `malloc` manages it.
- The **stack** is used every time a function is called or a local variable is created, growing upward — like stacking trays in a dining hall.

("Up" and "down" are an artist's rendition; the hardware has no notion of either.)

Each function call gets its own chunk of stack memory — technically a **frame** — holding its arguments and local variables. When `main` calls `swap`, `swap`'s frame is stacked on top of `main`'s. When `swap` returns, its frame is conceptually gone: the zeros and ones physically remain, but the computer is free to reuse that memory — *ergo garbage values*, the remnants §4.10 printed.

### 4.12.4 Tracing the Failed Swap

With frames in the picture, the failure traces cleanly. `main`'s frame holds `x = 1`, `y = 2`. Calling `swap(x, y)` builds a frame above it with copies: `a = 1`, `b = 2`, plus `tmp`.

| Step | Code | `a` | `b` | `tmp` | `x` | `y` |
|------|------|-----|-----|-------|-----|-----|
| start | `swap(x, y)` called | 1 | 2 | — | 1 | 2 |
| 1 | `int tmp = a;` | 1 | 2 | 1 | 1 | 2 |
| 2 | `a = b;` | 2 | 2 | 1 | 1 | 2 |
| 3 | `b = tmp;` | 2 | 1 | 1 | 1 | 2 |
| return | frame discarded | — | — | — | **1** | **2** |

The copies were swapped flawlessly. (Unlike the emptied glass, `tmp` still holds its 1 at the end — a copy remains, merely useless.) Nobody ever touched `x` or `y`, and before this lecture, no syntax existed to express a function that could.

### 4.12.5 Pass by Reference

The fix is to hand `swap` not the values but the **addresses** of `x` and `y` — a treasure map leading to the real variables. That is **passing by reference**. `swap1.c`:

```c
// Swaps two integers using pointers

#include <stdio.h>

void swap(int *a, int *b);

int main(void)
{
    int x = 1;
    int y = 2;

    printf("x is %i, y is %i\n", x, y);
    swap(&x, &y);
    printf("x is %i, y is %i\n", x, y);
}

void swap(int *a, int *b)
{
    int tmp = *a;
    *a = *b;
    *b = tmp;
}
```

```text
x is 1, y is 2
x is 2, y is 1
```

Three coordinated changes, all applications of this week's building blocks:

1. The parameters become `int *a, int *b` — each now *the address of* an `int`. (The prototype must change to match.)
2. The body dereferences: `*a` means "go to the address in `a`". When no data type stands to the left of the star, the star means *go there*.
3. The call site passes `&x, &y` — the addresses, not the values.

The trace, with `a` pointing at `x` and `b` pointing at `y`:

| Step | Code | `tmp` | `x` (via `*a`) | `y` (via `*b`) |
|------|------|-------|----------------|----------------|
| start | `swap(&x, &y)` called | — | 1 | 2 |
| 1 | `int tmp = *a;` — go to `a`'s target, fetch 1 | 1 | 1 | 2 |
| 2 | `*a = *b;` — go to `b`'s target, fetch 2, store it at `a`'s target | 1 | **2** | 2 |
| 3 | `*b = tmp;` — store 1 at `b`'s target | 1 | 2 | **1** |

When `swap` returns and its frame evaporates, the mutation survives, because it happened in `main`'s frame all along — chutes-and-ladders style, each dereference slid down the arrow into `main`'s memory.

> The star is now doing three jobs — multiplication, pointer declaration, dereference — and the overuse is genuinely confusing. Ideally the designers of C would have picked another symbol decades ago; the context (after a type name? before a pointer in an expression?) is what disambiguates.

### 4.12.6 Questions

1. `swap0.c`'s logic faithfully copies the glass-pouring procedure, yet `x` and `y` end unchanged — root cause?
2. The four regions of a running program's memory, with one occupant of each?
3. What is a frame, what happens to it on return, and which earlier phenomenon does that reuse explain?
4. The three coordinated changes that turn `swap0.c` into the working `swap1.c`?
5. Tracing `swap(&x, &y)` with `x = 1, y = 2`: the values of `tmp`, `x` and `y` after each of the three body lines?
6. Three different meanings of `*` seen this week, with one example each?

## 4.13 Overflow: When the Regions Collide

The memory map has a designed-in hazard: the heap grows downward, the stack grows upward, and the computer's memory is finite. Enough `malloc` calls (heap) or enough function calls without returns (stack) and the two regions collide.

There is no clever avoidance mechanism — the mitigation is simply restraint:

- allocate only the memory actually needed, not a million bytes "just in case";
- avoid calling functions again and again without them returning — the Week 3 recursion hazard of a missing base case, which piles up frames until the stack is exhausted.

The vocabulary, and one etymology:

| Term | Meaning |
|------|---------|
| **stack overflow** | too many frames — e.g. runaway recursion — exhaust the stack region |
| **heap overflow** | allocations exhaust the heap region |
| **buffer overflow** | the general case: a chunk of memory (often an array) gets more data than it can hold — e.g. a small array with too many values stuffed in |

The popular programmers' Q&A website Stack Overflow is named after exactly this failure mode.

### 4.13.1 Questions

1. The heap grows downward and the stack upward — what failure does this geometry permit, and what is the honest mitigation?
2. Which Week 3 mistake manufactures a stack overflow?
3. Buffer overflow, heap overflow, stack overflow — the relationship between the three terms?

## 4.14 Input Without the CS50 Library: scanf

The remaining training wheels are the CS50 input functions — `get_char`, `get_double`, `get_float`, `get_int`, `get_long`, `get_string` and friends. They are harder to take off than `string` was, because C fundamentally does not make user input easy. This section shows what they hide.

### 4.14.1 Replacing get_int

The baseline, `get0.c`, with the library:

```c
// Gets an int from user using get_int

#include <cs50.h>
#include <stdio.h>

int main(void)
{
    int n = get_int("n: ");
    printf("n: %i\n", n);
}
```

C's own alternative is **`scanf`** — *scan formatted* input: read something from the keyboard into memory. It shares `printf`'s format codes (`%i`, `%s`, …), just pointed in the other direction. `get1.c`:

```c
// Gets an int from user using scanf

#include <stdio.h>

int main(void)
{
    int n;
    printf("n: ");
    scanf("%i", &n);
    printf("n: %i\n", n);
}
```

The load-bearing character is the `&`. For `scanf` to *change* `n` — exactly like `swap` changing `x` — it must receive the **address** of `n`, not a copy of its (garbage) value. `scanf("%i", n)` would be the swap0 mistake all over again; `&n` is the treasure map. (The `printf("n: ")` line is merely the prompt, since `scanf` prints nothing itself.)

Typing 50 echoes 50 back: `get_int` replaced, library gone.

### 4.14.2 A Broken get_string

Strings are where the replacement falls apart. The naïve attempt, `get2.c`:

```c
// Incorrectly gets a string from user using scanf

#include <stdio.h>

int main(void)
{
    char *s;
    printf("s: ");
    scanf("%s", s);
    printf("s: %s\n", s);
}
```

Note the deliberate *absence* of `&` before `s` this time: `s` is already an address — that is what a `char *` is — and `%s` wants exactly the address at which to start storing characters. (`&s` would be the address of an address, a *pointer to a pointer* — a real thing, but not for today.)

The actual bug sits on the first line. `char *s;` reserves 8 bytes for an address but assigns nothing, so `s` contains a garbage address — 8 garbage bits × 8, pointing somewhere random, drawn in lecture as a squiggly arrow into the void. `scanf` then obediently stores the typed characters *at that random location*: memory that was never allocated, never controlled.

Typing something short like `hi!` may well *appear* to work — the program got lucky, nothing more. A long enough input would eventually crash it. valgrind is not fooled even when the run looks clean:

```text
Use of uninitialised value of size 8
Use of uninitialised value of size 8
...
```

— the size-8 uninitialised value being the pointer `s` itself.

### 4.14.3 Patches, and Where the Line Gets Drawn

Two ways to give `s` real memory:

```c
char *s = malloc(4);   // 4 bytes from the heap (plus NULL check, plus free)
```

```c
char s[4];             // get3.c: an array of 4 chars on the stack
```

The array version works with `scanf("%s", s)` unchanged, because an array's name acts as a pointer — the syntactic sugar of §4.6.3 running in the other direction: arrays may be treated as addresses and addresses as arrays.

Either way, typing `hi!` (3 characters + `\0` = exactly 4 bytes) fits. And either way the fundamental problem stands: **how many bytes is the human going to type?** `hi!` fits in 4; a longer word needs 5, 6, 7… maybe 100, maybe 10,000, maybe a million. At some point a line must be drawn in the sand — which is why web forms cap input lengths — and any input beyond it is again a write into unowned memory: unsafe, possibly a crash, at best luck.

`get_string` earns its keep precisely here. Its implementation takes baby steps through the input, calling `malloc` again and again — one more byte if needed, one more byte if needed — so that almost any length is handled and the caller never chooses a limit.

> The honest verdict from lecture: getting user input safely in raw C is a pain in the neck. The practical options are (a) a library like CS50's or (b) a language that solves it outright — Python, two weeks away, where these problems still exist but are abstracted away by code other humans already wrote.

### 4.14.4 Questions

1. `scanf("%i", &n)` requires the ampersand, while `scanf("%s", s)` must *not* have one — both facts follow from one principle. Which?
2. The bug in `get2.c` (`char *s; scanf("%s", s);`) — and why can a test run hide it?
3. The two patches that give `s` real memory, and the fundamental problem neither solves?
4. How does `get_string` sidestep the fixed-size problem?
5. What would `&s` denote, given `char *s`?

## 4.15 File I/O

### 4.15.1 RAM versus Persistent Storage

Every program written so far in the course has amnesia: even last week's phone book forgot every name and number the moment the program quit. The data lived only in RAM.

| | RAM (random-access memory) | Disk (HDD / SSD) |
|---|---|---|
| Role | where programs and files live *while in use* | where data lives *long term* |
| Speed | fast | slower |
| Power off | **everything is lost** — *volatile* | everything survives — *persistent / non-volatile* |

(A hard drive is the spinning-platter kind; an SSD — solid-state drive — has no moving parts.) A **file** is just a bunch of bytes stored on disk. **File I/O** (input/output) is the ability to create, read, write and persist files from code — what the File menu of Google Docs or Microsoft Word does, finally available in C.


### 4.15.2 The stdio File Functions

`stdio.h` — home of `printf` all along — also declares the file family, many prefixed with `f`:

| Function | Purpose |
|----------|---------|
| `fopen`  | open a file (returns a pointer to it) |
| `fclose` | close (and effectively save) a file |
| `fprintf`| `printf`, but into a file |
| `fscanf` | `scanf`, but from a file |
| `fread`  | read raw bytes from a file |
| `fwrite` | write raw bytes to a file |
| `fseek`  | jump to a position within a file |

### 4.15.3 A Persistent Phone Book

Target: a phone book whose entries survive the program, stored as a **CSV** (comma-separated values) file — a lightweight spreadsheet where each line is a row and commas separate the columns. `phonebook0.c`:

```c
// Saves names and numbers to a CSV file

#include <cs50.h>
#include <stdio.h>
#include <string.h>

int main(void)
{
    // Open CSV file
    FILE *file = fopen("phonebook.csv", "a");

    // Get name and number
    char *name = get_string("Name: ");
    char *number = get_string("Number: ");

    // Print to file
    fprintf(file, "%s,%s\n", name, number);

    // Close file
    fclose(file);
}
```

Piece by piece:

- **`FILE *file = fopen(...)`** — `fopen` returns a pointer to a `FILE`, a struct someone implemented years ago. The all-caps name is a historical quirk, not a constant. For all intents and purposes the pointer leads to the file's contents — technically to a data structure in memory that *references* the file on disk, but the white lie serves.
- **The mode string matters.** The lecture's first attempt used `"w"` (write) — and every run wiped the file, because `"w"` starts writing at the first byte, rewriting the file from scratch. Kelly appeared; running it again for a second entry erased her. The fix is `"a"` — **append** — which adds to the end instead. (The same read/write/append convention shows up in other languages too.)
- **`fprintf(file, ...)`** — identical to `printf` except the first argument names *which* file to print into. `%s,%s\n` produces exactly one CSV row per run: `David,617-495-1000`, then `Kelly,617-495-1000`, accumulating.
- **`fclose(file)`** — closes the file so it is effectively saved.

The resulting `phonebook.csv` is a real file: downloadable, double-clickable, openable in Microsoft Excel, Apple Numbers or Google Sheets.

Why pointers had to come first: this is simply how C designed files to work. Opening a file hands back an address; writing means telling `fprintf` which address to go to. Without pointers there is no file I/O in C — and hence no way to teach files in Week 1 or 2 without this "stupid little character" (`*`) meaning something.

### 4.15.4 Checking fopen

As with any function returning a pointer, `fopen` can fail — out of space, say — and return `NULL`. `phonebook1.c` adds the by-now-standard guard:

```c
    FILE *file = fopen("phonebook.csv", "a");
    if (file == NULL)
    {
        return 1;
    }
```

> The general rule crystallising across this lecture: **whenever dealing with pointers, check the return values.** `get_string`, `malloc`, `fopen` — each returns `NULL` when it cannot oblige, and each deserves an `if (... == NULL) return 1;`.

### 4.15.5 Questions

1. Volatile versus persistent storage — and which one held every previous program's data?
2. What does `fopen` hand back, and what obligation comes with it?
3. The bug observed with mode `"w"`, and the one-character fix?
4. How does `fprintf(file, "%s,%s\n", name, number)` produce a spreadsheet-openable file?
5. Why could file I/O not have been taught before pointers?

## 4.16 Buffers and a Home-Made cp

### 4.16.1 What "Buffering" Means

A **buffer** is just a chunk of memory — in C, typically an array of finite size — that stores bytes of stuff in transit.

A video player's progress bar is the everyday example: the buffer holds the next few bytes of the video, downloaded ahead of the playhead. With a fast connection, bytes arrive faster than they are watched; with a slow one, the playhead catches up to the end of the buffer and the app announces *buffering* — the word now demystified.

### 4.16.2 cp.c: Copying a File Byte by Byte

The same byte-at-a-time pattern, aimed at copying instead of watching — a home-made version of the `cp` command used in the terminal for weeks. `cp.c`:

```c
// Copies a file

#include <stdio.h>

typedef unsigned char BYTE;

int main(int argc, char *argv[])
{
    FILE *src = fopen(argv[1], "rb");
    FILE *dst = fopen(argv[2], "wb");

    BYTE b;

    while (fread(&b, sizeof(b), 1, src) != 0)
    {
        fwrite(&b, sizeof(b), 1, dst);
    }

    fclose(dst);
    fclose(src);
}
```

New pieces, one at a time:

- **`char *argv[]`** — the final white lie retired. `main`'s second parameter was declared `string argv[]` all these weeks; this is what it really was. `argv[1]` is the source file name, `argv[2]` the destination (`argv[0]` being the program's own name, as ever).
- **`typedef unsigned char BYTE;`** — C has no built-in type named "byte", but `typedef` (§4.5.3) manufactures one. A `char` is exactly 1 byte; **`unsigned`** tells the compiler these 8 bits are never to be interpreted as a negative number — they are raw data, not math material.
- **`"rb"` / `"wb"`** — read and write in *binary* mode. For files known to be raw zeros and ones (images, say) rather than ASCII/Unicode text, the `b` prevents the bytes being mistaken for text.
- **`fread(&b, sizeof(b), 1, src)`** — read, into the address of `b`, items of size `sizeof(b)` (= 1 byte), 1 item at a time, from `src`. Its return value is how many items were actually read, so `!= 0` means "while a byte was successfully read".
- **`fwrite(&b, sizeof(b), 1, dst)`** — the mirror image: write that 1 byte to `dst`.
- **`fclose` both files** — destination saved, source released.

The loop is the progress bar in miniature: read a byte, write a byte, read a byte, write a byte, until `fread` comes up empty. A huge mouthful admittedly, but the shape recurs in the problem set.

```text
$ make cp
$ ./cp phonebook.csv copy.csv
```

Opening `copy.csv` shows David and Kelly intact — a byte-for-byte copy made with syntax unavailable until today.

> Nitpick owed by §4.15.4: production code would check both `fopen` calls (and even `fwrite`) for failure before proceeding. Omitted here for brevity, not correctness.

### 4.16.3 Questions

1. What is a buffer, and what is a video player doing when it says "buffering"?
2. Why `typedef unsigned char BYTE;` — both halves of the definition?
3. `fread(&b, sizeof(b), 1, src)` — the meaning of each argument, the return value, and how the copy loop terminates?
4. With the training wheels off, what is `argv` really?

## 4.17 Looking Ahead: Bitmaps, Filters and Beyond

The pixel-art preview (§4.1) now closes the loop. A **BMP** (bitmap) file is essentially a grid of pixels stored top to bottom, left to right — literal sequences of pixels, each represented by a red value, a green value and a blue value (§4.2.1).

With file I/O and byte-level control, the problem set has code iterate over such files and mutate the bytes — Instagram-style filters, hand-rolled:

| Filter | Byte-level idea |
|--------|-----------------|
| Greyscale | shrink each pixel's R, G and B toward equal grey tones |
| Sepia | recompute R/G/B for an old-photograph tint |
| Reflect | move the bytes from one side to the other, mirroring over the vertical axis |
| Blur | make every pixel a little fuzzier via math over its neighbours |
| Edge detection | find the salient boundaries — a bridge against the sky |

JPEGs get similar treatment in the weeks ahead.

The motivating question — *why does any of this memory detail matter?* — got a two-part answer in lecture. Files are one part: no pointers, no file I/O. The other arrives next week: building two-dimensional structures in memory — family trees, and trees generally — that store data more efficiently than anything an array allows. Arrays enable binary search, but some things arrays cannot do, especially when speed matters.

### 4.17.1 Questions

1. At byte level, what is a BMP, and which stored values do the problem set's filters mutate?
2. The lecture's two-part answer to "why does any of this memory detail matter"?

## 4.18 Cheat Sheet — Week 4 in One Place

### 4.18.1 Operators

| Syntax | Meaning |
|--------|---------|
| `&x` | address of `x` |
| `int *p` | declare `p` as pointer to `int` |
| `*p` | dereference: go to the address in `p` |
| `s[i]` ≡ `*(s + i)` | array sugar ↔ pointer arithmetic (both directions) |
| `sizeof(type)` | size in bytes of a type on this system (`char` 1, `int` usually 4, pointers 8) |
| `typedef old new;` | create type synonym (`typedef char *string;`, `typedef unsigned char BYTE;`) |
| `0x…` | the number that follows is hexadecimal |
| `%p` | printf/scanf format code for an address |

### 4.18.2 Functions

| Function | Header | Job | Gotcha |
|----------|--------|-----|--------|
| `malloc(n)` | `stdlib.h` | allocate `n` bytes on the heap; returns address of first byte | returns `NULL` on failure; must be freed |
| `free(p)` | `stdlib.h` | return `malloc`'d memory | only for memory obtained via `malloc`; not `get_string`'s |
| `strcmp(s, t)` | `string.h` | compare strings char by char | equality is return value **0** |
| `strcpy(dst, src)` | `string.h` | copy string incl. `\0` | destination **first**; dst must be big enough |
| `strlen(s)` | `string.h` | visible length | excludes `\0` — allocate `strlen(s) + 1`; hoist out of loop conditions |
| `scanf(fmt, addr)` | `stdio.h` | read formatted input to an address | `&n` for an `int`; a `char *` must already point at real memory |
| `fopen(name, mode)` | `stdio.h` | open file; modes `"r"`, `"w"` (overwrite!), `"a"` (append), + `b` for binary | returns `NULL` on failure |
| `fclose(f)` | `stdio.h` | close/save file | — |
| `fprintf(f, fmt, …)` | `stdio.h` | printf into a file | file argument first |
| `fread(&b, size, count, f)` | `stdio.h` | read raw bytes | returns items read; 0 = done |
| `fwrite(&b, size, count, f)` | `stdio.h` | write raw bytes | mirrors `fread` |
| `valgrind ./prog` | (tool) | find memory errors and leaks | look for the file:line breadcrumbs |

### 4.18.3 Memory Map

```text
+----------------------+
|     machine code     |
+----------------------+
|       globals        |
+----------------------+
|      heap  ↓         |   malloc lives here
|                      |
|      stack ↑         |   frames: functions + locals
+----------------------+
```

### 4.18.4 Classic Bugs of Week 4

| Bug | Symptom | Fix |
|-----|---------|-----|
| `s == t` on strings | compares addresses, always "Different" | `strcmp(s, t) == 0` |
| `t = s` to "copy" a string | alias — edits hit both | `malloc(strlen(s) + 1)` + `strcpy` |
| loop `i < strlen(s)` when copying | `\0` not copied | `<=`, with `n = strlen(s)` hoisted |
| no `free` after `malloc` | leak — "definitely lost: N bytes" | `free(p)` when done |
| no `NULL` check on `get_string`/`malloc`/`fopen` | dereference of `0x0` on failure | `if (p == NULL) return 1;` |
| dereferencing an uninitialised pointer | crash / corruption (Binky) | point it at real memory first |
| `x[3]` on a 3-int allocation | "Invalid write of size 4" | indices 0, 1, 2 |
| `scanf("%s", s)` with bare `char *s` | writes to a garbage address | array / `malloc` — better: a library |
| `fopen(..., "w")` for repeated saves | file rewritten from byte 0 each run | `"a"` to append |
| `toupper(t[0])` on empty string | touches the `\0` | guard `strlen(t) > 0` |

### 4.18.5 Questions

1. A program gets a string from the user, makes a capitalised copy, appends it to a file and exits cleanly. Drawing on the whole lecture: which checks and cleanups does it owe?
2. The four distinct roles played by `*` and `&` this week, each with a minimal example?

## 4.19 Answers

Worked answers to every `Questions` subsection. Each `###` below reuses the
number of the section it answers rather than the hierarchical counter (the
authoring spec's §5.1 exception), so the answers to `### 4.4.8 Questions` sit
under `### 4.4 Answers`. Answer numbers pair with question numbers.

### 4.1 Answers

**1.** Resolution gives the pixel count: 800 × 600 = 480,000 pixels. At 24 bits — 3 bytes — per pixel, the data is 480,000 × 3 = 1,440,000 bytes, roughly 1.4 megabytes. The two facts: resolution is horizontal dots × vertical dots, and the bits spent per pixel determine the size of each dot's colour description.

**2.** One bit has exactly two states, so each pixel can only be black (`0`) or white (`1`) — the smiley grid's entire vocabulary. Spending more bits per pixel multiplies the distinct patterns available: 24 bits allows 2²⁴ ≈ 16.7mn combinations, which is how photographs get every shade of the rainbow.

### 4.2 Answers

**1.** In a two-digit hex number the right column is the 1s place and the left the 16s place. `A` is 10, so `0xA5` = 16 × 10 + 1 × 5 = 160 + 5 = **165**.

**2.** Sixteen is 2⁴, so the sixteen hex digits `0`–`F` map one-to-one onto the sixteen possible patterns of 4 bits (`0000` through `1111`). `F` = 15 = `1111` tops out both ranges at once, which is why the mapping is exact rather than approximate.

**3.** `0x80` = 16 × 8 + 0 = **128** — almost exactly half of the channel maximum 255.

**4.** In base 16 the carry happens after `F` (fifteen), not after `9`, so the numeral `10` means 1 × 16 + 0 × 1 = sixteen. The `0x` prefix marks the base, making `0x10` unambiguous at a glance.

**5.** Full red is `FF`, half green is `80` (128 of 255, per question 3), no blue is `00`: **`FF8000`** — an orange.

### 4.3 Answers

**1.** Four bytes. An `int` is 32 bits by convention on most systems regardless of the value stored; 50 needs only a sliver of that range, but the full 4 bytes are reserved.

**2.** Because of the 4-bits-per-digit mapping (§4.2.3): every byte is exactly two hex digits, so addresses align neatly with the unit memory is organised in. Decimal offers no such alignment.

**3.** The name `n` stands for a location. `printf` receives the value found by going to `n`'s address (the diagrams' `0x123`), reading the 4 bytes there and formatting them as a decimal integer. Programs have performed this address lookup invisibly since Week 1.

### 4.4 Answers

**1.** A pointer is a variable that stores an address — the location of some other value in memory.

**2.** `int *` declares the variable's type as *address of an int*; `p` is the variable's name; `&n` produces the address at which `n` lives, which becomes the value stored in `p`.

**3.** A 32-bit pointer can name only about 4bn distinct addresses — 4 gigabytes' worth. Machines with 8, 16 or more gigabytes need bigger house numbers, so 64 bits (8 bytes) became the convention.

**4.** It prints `50`. Trace: `n`'s 4 bytes hold 50 at, say, `0x123`; `p` holds `0x123`; `*p` follows that address, reads the 4 bytes found there and hands the 50 to `printf`.

**5.** Compilation fails with **"incompatible pointer to integer conversion"**. Types are promises about meaning, not just size: an `int` is data for math, a pointer is a location. Allowing one to silently become the other would hide exactly the class of bug this whole lecture is about.

**6.** The label is the variable's name; the slip of paper inside `p`'s box is the address `0x123`; the foam finger is the dereference — physically following the stored address to the box where the 50 actually lives.

### 4.5 Answers

**1.** `s` stores one address — that of the first character `H` (`0x123` in the diagrams). The characters themselves live elsewhere in memory, as 4 contiguous bytes: `H`, `I`, `!`, `\0`.

**2.** Two conventions cooperate: a string's characters are contiguous (no gaps), and every string ends in the null terminator. The pointer marks the beginning, `\0` marks the end, and a loop finds everything in between — so storing every character's address would be redundant.

**3.** It creates a type synonym: wherever `string` appears, the compiler reads `char *`. The library shipped it so that Weeks 1–3 could use strings without first teaching hexadecimal, addresses, pointers and dereferencing.

**4.** `s[0]` is a `char` — a value — so obtaining its location requires the address-of operator. `s` needs no `&` because it already *is* an address; that is the entire reveal of this section.

**5.** Given `%s` and an address, `printf` loops: print the character at the address, advance one byte, repeat until the byte read is `\0`, then stop. `%c` performs no loop — one character, done.

### 4.6 Answers

**1.** `*(s + 2)` — take the address in `s`, move 2 bytes along, go there.

**2.** `I!` — `s + 1` is the address of the `I`, and `%s` prints from there until the shared null terminator.

**3.** Operator precedence. `*s + 1` dereferences first and then adds 1 to the *character's code*, a piece of character arithmetic rather than a move through memory — a different operation that only coincidentally gives the same letter here (the code after `H` is `I`). The parentheses force the pointer arithmetic to happen before the dereference.

**4.** Nicer syntax for an operation the language can already express less readably. `s[i]` compiles to exactly `*(s + i)`; the brackets add no capability, only legibility — and the equivalence runs in both directions, as §4.14.3's array-passed-as-pointer shows.

### 4.7 Answers

**1.** The two addresses. `s == t` asks whether `0x123` equals `0x456` — whether the pointers name the same location — not whether the bytes stored at those locations match. The comparison answers its question correctly; it is simply the wrong question.

**2.** `0` on equality — hence the idiom `strcmp(s, t) == 0`. The non-zero returns signal ordering: which string sorts before or after the other, which also makes `strcmp` usable for alphabetising.

**3.** `get_string` allocates fresh memory on every call and performs no lookup for previous identical input. Being "really smart and generous" about reuse is simply not how it works: same text, two allocations, two addresses.

**4.** When both pointers hold the same address — for instance after the aliasing assignment `t = s` of §4.8.1. Then `s == t` is true precisely because only one string exists in memory.

### 4.8 Answers

**1.** `t = s` copies the address, not the characters, so both names point at the same 4 bytes. `t[0]` and `s[0]` are therefore the same physical byte; capitalising it changes the one string that both variables share. (In R, `t <- s` behaves like a copy and this bug cannot occur.)

**2.** `strlen` counts only the visible characters — 3 for `hi!` — but the copy also needs its null terminator. Without the extra byte the terminator has nowhere to go, and the "string" never officially ends.

**3.** The `<=` runs the loop one extra iteration, copying `s[3]` — the `\0` itself — into `t[3]`. With `<`, the copy would be unterminated; patching it afterwards with a hard-coded `t[3] = '\0'` works but is sloppier than letting the loop do it.

**4.** `strlen(s)` is recomputed on every iteration although the answer never changes. The `copy3.c` fix hoists it into the initialiser: `for (int i = 0, n = strlen(s); i <= n; i++)` — the length is computed once.

**5.** Destination first, then source: `strcpy(t, s)` copies *into* `t`. Reversed, the not-yet-initialised copy would be written over the original — destroying the very data being copied.

**6.** NUL is the null *character*, `\0` — a single all-zero byte that terminates a string. NULL is the null *address*, `0x0` — a location deliberately kept empty so it can serve as the "something went wrong" sentinel returned by `get_string`, `malloc` and `fopen`.

**7.** A leak is `malloc`'d memory never freed: the computer believes it is still in use, so available memory shrinks for as long as the program runs. Long-running leaky programs get slower and slower — the classic symptom on phones and desktops alike. `t` came from `malloc`, so freeing it is the caller's job; `s` came from `get_string`, whose CS50 implementation frees its own allocations automatically, so `free(s)` would meddle with memory the library manages.

### 4.9 Answers

**1.** Portability and legibility. `sizeof(int)` asks for the type's actual size on the system at hand, whereas 12 bakes in the assumption that an `int` is 4 bytes everywhere. The expression also states the intent — space for three ints — instead of a magic number.

**2.** Line 11 is `x[3] = 33;`. A 3-int allocation has valid indices 0, 1 and 2, so `x[3]` *writes* — changes a value, the counterpart of a read — 4 bytes (one `int`'s worth, hence "size 4") into memory beyond the allocation.

**3.** By program exit, 12 bytes allocated in a single `malloc` call ("1 blocks") were never freed: 3 ints × 4 bytes = 12. Valgrind's companion breadcrumb names line 8 — the `malloc` itself — as the origin of the lost block, pointing straight at what needs a matching `free`.

**4.** The bug is latent: the out-of-bounds write happened to land somewhere whose corruption caused no visible failure on this run. The absence of a crash means the program got lucky, not that it is correct — precisely the class of bug valgrind exists to expose.

### 4.10 Answers

**1.** A garbage value is whatever bit pattern already occupied a piece of memory before the current code claimed it — remnants of earlier computation, not values placed there deliberately. The loop prints the previous contents of the 4096 bytes the array happens to cover: stray zeros, a 25, a 32000-ish value, negative numbers.

**2.** Initialisation. `numeric(1024)` hands back 1024 genuine zeros; C reserves the space and stops there, leaving the old contents in place for speed. Filling the array is the programmer's job in C, and reading it before filling it is the bug.

### 4.11 Answers

**1.** `x` was aimed at a real pointee by `malloc`, so dereferencing it lands on owned memory. `y` was declared but never aimed at anything: it holds a garbage address, and `*y` jumps to that random location — the step that felled Binky. Allocating a pointer and setting up its pointee are separate steps, and `y`'s second step never happened.

**2.** No value moves. Pointer assignment copies the *address*, so `y` comes to point at the same pointee as `x` — two arrows, one box. The 42 sits in that box untouched throughout the assignment.

**3.** Because there is only one pointee. `*y = 13` overwrites the shared box's contents, so `*x` — a different route to the same box — now reads 13 as well. Sharing is both the point of the fix and the punchline of the claymation.

### 4.12 Answers

**1.** Pass by value. C hands `swap` *copies* of `x` and `y`: `a` and `b` live in `swap`'s own frame, a different scope from `main`'s. The copies are swapped flawlessly and discarded on return; nothing in `main`'s frame was ever touched. (An R function that reassigns its arguments changes nothing outside itself either — the difference is that C offers the pointer escape hatch.)

**2.** Machine code — the program's own compiled 0s and 1s; globals — variables declared outside every function; the heap — `malloc`'s territory, growing downward; the stack — function frames and local variables, growing upward like dining-hall trays.

**3.** A frame is the chunk of stack memory a function call receives for its arguments and locals. On return the frame is not erased — the bits physically remain — but the region becomes reusable and the next call overwrites it. Those leftovers are exactly the garbage values of §4.10.

**4.** Parameters become addresses (`int *a, int *b`, prototype included); the body dereferences (`int tmp = *a; *a = *b; *b = tmp;`); the call site passes addresses (`swap(&x, &y)`).

**5.** After `int tmp = *a;`: `tmp` 1, `x` 1, `y` 2. After `*a = *b;`: `tmp` 1, `x` **2**, `y` 2. After `*b = tmp;`: `tmp` 1, `x` 2, `y` **1**. Every write travelled through the arrows into `main`'s frame, which is why the effect survives `swap`'s return.

**6.** Multiplication (`a * b`); pointer declaration — star after a type (`int *p`); dereference — star before a pointer in an expression (`*p = 13`). One symbol, three jobs; whether a type name stands to the left is the disambiguator.

### 4.13 Answers

**1.** Collision: with finite total memory, enough allocations from above and enough simultaneous function calls from below eventually meet. No mechanism prevents it — the mitigation is restraint: allocate only what is needed, and avoid calling functions endlessly without returns.

**2.** Recursion without a working base case. Each call stacks another frame without ever shrinking the problem, and the frames pile up until the stack region is exhausted — the etymology of the Stack Overflow website's name.

**3.** Buffer overflow is the general term: a finite chunk of memory (often an array) receiving more data than it can hold. Stack overflow and heap overflow are the region-specific cases — the respective region running out of room.

### 4.14 Answers

**1.** Functions receive copies, so changing a caller's variable requires its address — the `swap` lesson. `n` is an `int`, so its address must be taken explicitly with `&`. `s` is *already* an address (that is what a `char *` is), and `%s` wants exactly that: the location at which to start storing characters.

**2.** `char *s;` reserves 8 bytes for an address but never aims them, so `s` holds a garbage address and `scanf` writes the typed characters to that random location. A short input may land harmlessly — luck, not correctness; valgrind reports "use of uninitialised value of size 8" (the 8-byte pointer itself) even on a run that looks clean.

**3.** `char *s = malloc(4);` — heap memory, owing a NULL check and a later `free` — or `char s[4];`, a stack array that works unchanged because an array's name acts as a pointer. Neither answers the real question: how many bytes will the human type? Any fixed line in the sand can be exceeded, and past it the writes land in unowned memory again.

**4.** By refusing to guess. Its implementation takes baby steps through the input, calling `malloc` again and again — one more byte whenever needed — so the buffer grows to fit almost any input and the caller never chooses a limit.

**5.** The address of the pointer variable itself — a pointer to a pointer (`char **`). A real construct, explicitly deferred by the lecture: "none of that today".

### 4.15 Answers

**1.** RAM is volatile — its contents vanish with power (or with the program's exit); disk, whether spinning-platter hard drive or SSD, is persistent. Every program before this lecture kept its data only in RAM, which is why last week's phone book forgot everything at exit.

**2.** A `FILE *` — a pointer to a `FILE` struct that, for practical purposes, leads to the file (technically to an in-memory structure referencing the file on disk). Like every pointer-returning function, `fopen` can return `NULL` — out of space, say — so the return value deserves a check before any use.

**3.** `"w"` writes starting at the first byte — a full rewrite — so each run of the phone book erased the previous entries; Kelly vanished on the next launch. `"a"` appends to the end instead, letting the rows accumulate.

**4.** Each run writes one line: the two `%s` values separated by a comma, closed with a newline — `David,617-495-1000`. Lines as rows, commas as column separators: exactly the CSV convention that Excel, Apple Numbers and Google Sheets parse as a spreadsheet.

**5.** Because C's file interface *is* pointers: opening a file yields an address, and every subsequent read or write names that address. Teaching `fopen` in Week 1 would have required explaining the "stupid little character" `*` before variables had barely settled.

### 4.16 Answers

**1.** A finite chunk of memory — in C, typically an array — holding bytes in transit. A video player's buffer stores the next few not-yet-watched bytes of the video; "buffering" means playback has caught up with downloading and the buffer has run dry.

**2.** `char` supplies the size: exactly 1 byte, 8 bits. `unsigned` strips away negative interpretation — these are raw bits, not numbers destined for math, so a sign would only mislead. `typedef` then names the combination `BYTE`, a type C never shipped but §4.5.3's tool can manufacture.

**3.** Read into the address of `b` (`&b`), items of size `sizeof(b)` — 1 byte — 1 item at a time, from `src`. The return value is the count of items actually read: 1 while bytes remain, 0 at end of file. `while (... != 0)` therefore runs exactly once per byte and stops precisely when the source is exhausted.

**4.** `char *argv[]` — an array of `char *`, one pointer per word typed at the command line. The `string argv[]` written since Week 2 was the same declaration wearing the library's `typedef`.

### 4.17 Answers

**1.** A grid of pixels stored top to bottom, left to right — each pixel a red value, a green value and a blue value, §4.2.1's triples now living in a file. Greyscale, sepia, reflect, blur and edge detection are all byte edits over those R/G/B values.

**2.** Files this week — without pointers there is no file I/O in C — and data structures next week: two-dimensional structures in memory (trees, family trees) that outgrow what arrays can do, especially where speed matters.

### 4.18 Answers

**1.** Check `get_string`'s return against `NULL`; check `malloc`'s return against `NULL`; allocate `strlen(s) + 1` bytes and copy with `strcpy(t, s)`; guard the capitalisation with `strlen(t) > 0` (an empty string's first byte is the terminator); open with `fopen(..., "a")` — not `"w"` — and check *that* against `NULL`; `fprintf` then `fclose`; `free(t)` but never `free(s)`; `return 1` on each failure path and `return 0` on success.

**2.** Address-of: `&x` produces a variable's location. Pointer declaration — star after a type: `int *p` stores an address. Dereference — star before a pointer in an expression: `*p = 13` goes there. Multiplication: `a * b`, the star's original job. Distinguishing them is context: a type name to the left means declaration; a pointer operand means dereference; two numeric operands mean math.